In [ ]:
import skimage as sk
import os
import numpy as np
import pandas as pd
import re
from cellpose import models, io
from random import randint
from glob import glob
import napari

In [ ]:
def get_center_project(file):
    img = sk.io.imread(file)
    center_z = img.shape[1]//2
    mid_slices = img[:,center_z-1:center_z+2,...]
    median_cent = np.median(mid_slices,axis=1)
    if img.shape[-1] == 1024:
        md_gauss = sk.filters.gaussian(median_cent,2)
    else:
        md_gauss = sk.filters.gaussian(median_cent,1)
    return median_cent, md_gauss, median_cent[0,...]

def process_nuc_channel(nuc_channel):
    sigma = nuc_channel.shape[-1]//3
    gauss = sk.filters.gaussian(nuc_channel,sigma)
    norm = sk.exposure.rescale_intensity(nuc_channel / gauss,out_range=(0,256))
    if nuc_channel.shape[-1] == 1024:
        sm_gauss = sk.filters.gaussian(norm,sigma=1)
        footprint = sk.morphology.disk(14)
        top_hat = sk.morphology.white_tophat(sm_gauss,footprint)
        md_gauss = sk.filters.gaussian(top_hat,4)
    else:
        sm_gauss = sk.filters.gaussian(norm,sigma=0.5)
        footprint = sk.morphology.disk(7)
        top_hat = sk.morphology.white_tophat(sm_gauss,footprint)
        md_gauss = sk.filters.gaussian(top_hat,2)        
    return md_gauss

def run_cp(img,cell_prob=0.0,flow=0.4,size=15):
    if img.shape[-1] == 1024:
        masks, _, _ = model.eval(img, rescale=0.5, resample=True, normalize=True, flow_threshold=flow, cellprob_threshold=cell_prob,min_size=size)
    else:
        masks, _, _ = model.eval(img, normalize=True, flow_threshold=flow, cellprob_threshold=cell_prob,min_size=size)
    return masks

def _filter_cells(cleared_cells,relabled_nuc):
    cell_props = sk.measure.regionprops(cleared_cells,relabled_nuc)
    cells_no_nuc = []
    cell_indicies = np.arange(np.max(cleared_cells)+1)
    for c in cell_props:
        label = c.label
        if c.intensity_max == 0:
            cells_no_nuc.append(label)
    #cells_no_nuc_indicies = np.arange(cells_no_nuc)
    cells_to_mask = np.isin(cell_indicies, cells_no_nuc)
    cells_mask = cells_to_mask[cleared_cells.reshape(-1)].reshape(cleared_cells.shape)
    cleared_cells[cells_mask] = 0
    relabled_cells, _, _ = sk.segmentation.relabel_sequential(cleared_cells)
    return relabled_cells

def clear_object_boarders(cell_masks,nuc_masks,name,path):
    cleared_cells, _, _ = sk.segmentation.relabel_sequential(sk.segmentation.clear_border(cell_masks))
    binary_cells = np.array(cleared_cells > 0,dtype="bool")
    masked_nuc = nuc_masks * binary_cells
    masked_nuc[masked_nuc>0] = 1
    matched_cells_to_nuc = _filter_cells(cleared_cells,masked_nuc)
    cell_bounaries = sk.segmentation.find_boundaries(matched_cells_to_nuc)
    masked_boundaries = np.array(matched_cells_to_nuc > 0,dtype='bool')
    masked_boundaries[cell_bounaries] = False
    if cell_masks.shape[-1] == 1024:
        footprint = sk.morphology.disk(2)
    else:
        footprint = sk.morphology.disk(1)
    eroded_masked_boundaries = sk.morphology.erosion(masked_boundaries,footprint)
    relabeled_nuc = (masked_nuc * eroded_masked_boundaries)*matched_cells_to_nuc # using eroded cells to crop nuclei within the cell boundary and minimize/remove overlaping nuclei labels within neighboring cells
    sk.io.imsave(os.path.join(path,f"{name}_cell_masks.tif"),matched_cells_to_nuc,check_contrast=False)
    sk.io.imsave(os.path.join(path,f"{name}_nuclei_masks.tif"),relabeled_nuc,check_contrast=False)
    return matched_cells_to_nuc, relabeled_nuc


def cyto_objects(cells, nuc, name,path):
    label_cyto = cells - nuc
    sk.io.imsave(os.path.join(path,f"{name}_cyto_masks.tif"),label_cyto,check_contrast=False)
    return label_cyto

def get_measurements(label_cyto,label_cells,label_nuc,img,name):
    props = ["label","intensity_median"]
    reshape_img = np.moveaxis(img,0,-1)
    cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_img,properties=props))
    nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_nuc,reshape_img,properties=props))
    cell_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cells,reshape_img,properties=props))
    cyto_props = cyto_props.add_prefix("Cyto_")
    nuc_props = nuc_props.add_prefix("Nuc_")
    cell_props = cell_props.add_prefix("Cell_")
    cyto_props["Nur77_Nuc_Cyto_Ratio"] = nuc_props["Nuc_intensity_median-1"]/cyto_props["Cyto_intensity_median-1"]
    merged = pd.concat([cell_props,nuc_props,cyto_props],axis=1)
    merged['Image'] = name
    return merged

def concat_measurements(dfs):
    return pd.concat(dfs,ignore_index=True)

In [ ]:
files = sorted(glob("/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Exported_Tiffs/*.tif"))
names = [os.path.basename(f)[:-4] for f in files]

In [ ]:
i = randint(0,len(files))
print(i)
test_img = sk.io.imread(files[i])
test_img.shape

In [ ]:
center_z = test_img.shape[1]//2

In [ ]:
mid_slices = test_img[:,center_z-1:center_z+2,...]

In [ ]:
mid_slices.shape

In [ ]:
model = models.CellposeModel(gpu=True)

In [ ]:
#i = randint(0,len(files))
masks_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Output_Masks/"
n = 'test_center_proj_and_norm'
test_center_proj, median_cent, nuc_channel = get_center_project(files[i])
test_nuc_norm = process_nuc_channel(nuc_channel)
test_cell_masks = run_cp(median_cent,flow=0.6,size=100)
test_nuc_masks = run_cp(test_nuc_norm,cell_prob=-1.0,flow=0.6)


In [ ]:
#test_object_boarders = sk.segmentation.find_boundaries(test_cell_masks,mode="inner")
test_cleared_cells, test_cleared_nuc = clear_object_boarders(test_cell_masks,test_nuc_masks,n,masks_save_path)
test_label_cyto = cyto_objects(test_cleared_cells, test_cleared_nuc,n,masks_save_path)

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(test_center_proj,channel_axis=0)
v.add_image(test_nuc_norm)
v.add_labels(test_cleared_cells)
v.add_labels(test_cleared_nuc)
#v.add_labels(eroded_masked_boundaries)
v.add_labels(test_label_cyto)

In [17]:
cyto_props, nuc_props, cell_props,merged = get_measurements(test_label_cyto,test_cleared_cells,test_cleared_nuc,median_cent,n)

In [19]:
merged_test = pd.concat([cyto_props, nuc_props, cell_props],axis=1)

In [20]:
merged_test

,Cyto_label,Cyto_intensity_median-0,Cyto_intensity_median-1,Cyto_intensity_median-2,Nur77_Nuc_Cyto_Ratio,Image,Nuc_label,Nuc_intensity_median-0,Nuc_intensity_median-1,Nuc_intensity_median-2,Image,Cell_label,Cell_intensity_median-0,Cell_intensity_median-1,Cell_intensity_median-2,Image
0,1,65.510072,29.792933,9.699142,2.452576,test_center_proj_and_norm,1,165.591606,73.069441,18.196163,test_center_proj_and_norm,1,128.792503,57.183754,15.058584,test_center_proj_and_norm
1,2,9.495752,15.087804,25.702372,2.707270,test_center_proj_and_norm,2,76.336522,40.846757,26.871390,test_center_proj_and_norm,2,26.208978,22.110664,26.706069,test_center_proj_and_norm
2,3,9.451536,14.283265,25.407506,2.930317,test_center_proj_and_norm,3,80.167369,41.854495,25.579276,test_center_proj_and_norm,3,28.202931,24.051176,25.507589,test_center_proj_and_norm
3,4,11.636214,13.596152,20.712351,3.171642,test_center_proj_and_norm,4,79.237384,43.122125,28.753706,test_center_proj_and_norm,4,52.959864,32.629321,27.479205,test_center_proj_and_norm
4,5,10.014706,14.798150,26.420672,2.748463,test_center_proj_and_norm,5,74.917065,40.672173,26.622835,test_center_proj_and_norm,5,45.799645,29.788337,26.617401,test_center_proj_and_norm
5,6,15.709316,18.580951,26.291558,3.229566,test_center_proj_and_norm,6,99.041491,60.008413,35.522488,test_center_proj_and_norm,6,58.945990,38.177039,33.300613,test_center_proj_and_norm
6,7,10.176192,15.884675,28.464738,2.306152,test_center_proj_and_norm,7,65.748165,36.632473,26.871277,test_center_proj_and_norm,7,37.524852,26.600160,27.147905,test_center_proj_and_norm
7,8,20.864767,21.122904,29.514629,2.648698,test_center_proj_and_norm,8,99.233104,55.948190,32.057489,test_center_proj_and_norm,8,77.933260,45.721524,31.516580,test_center_proj_and_norm
8,9,14.373986,16.683527,26.959183,2.544214,test_center_proj_and_norm,9,75.093924,42.446460,30.755240,test_center_proj_and_norm,9,43.347574,30.587514,29.839423,test_center_proj_and_norm
9,10,18.170596,20.719878,31.889747,2.297744,test_center_proj_and_norm,10,92.110329,47.608984,29.079372,test_center_proj_and_norm,10,68.618866,40.715861,29.396852,test_center_proj_and_norm


In [ ]:
v.add_image(test_img,channel_axis=0)

In [ ]:
join, map1, map2 = sk.segmentation.join_segmentations(test_cleared_cells,test_cleared_nuc,return_mapping=True)

In [ ]:
v.add_labels(join)

In [ ]:
#Need to add in preprocessing to remove background DAPI interferring with segmentation
# Testing adaptive hist
test_img = sk.io.imread(files[i])
#sigma = test_center_proj.shape[-1]//3
adapt_hist_proj = sk.exposure.equalize_adapthist(test_center_proj/255.0)

In [ ]:
# Testing background subtraction with rolling ball
background = sk.restoration.rolling_ball(test_center_proj[0],radius=5)
background_sub = test_center_proj[0] - background

In [ ]:
# Testing background norm with gaussian
sigma = test_center_proj.shape[-1]//3
gauss = sk.filters.gaussian(test_center_proj[0],sigma)
norm = sk.exposure.rescale_intensity(test_center_proj[0] / gauss,out_range=(0,256))

In [ ]:
sm_gauss = sk.filters.gaussian(norm,sigma=1)

In [ ]:
footprint = sk.morphology.disk(12)
top_hat = sk.morphology.white_tophat(sm_gauss,footprint)

In [ ]:
v.add_image(top_hat,name='tophat_12')

In [ ]:
mask, _, _ = model.eval(top_hat,normalize=True)

In [ ]:
v.add_labels(mask,name='cp_mask_tophat_12')

In [ ]:
#threshold and mask on norm
mask = sm_gauss > sk.filters.threshold_multiotsu(sm_gauss,classes=5)[-2]


In [ ]:
v.add_image(sm_gauss)
v.add_labels(mask)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
all_thresh = sk.filters.try_all_threshold(norm,figsize=(10,6))
plt.show()

In [ ]:
v.add_image(mask)

In [ ]:
model = models.CellposeModel(gpu=True)
masks_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Output_Masks/"
cyto_dfs = []
nuc_dfs = []
cell_dfs = []
for f, n in zip(files,names):
    center_proj = get_center_project(f)
    cell_masks = run_cp(center_proj)
    nuc_masks = run_cp(center_proj[0,...])
    object_boarders = sk.segmentation.find_boundaries(cell_masks,mode="inner")
    cleared_cells, cleared_nuc = clear_object_boarders(cell_masks,nuc_masks,n,masks_save_path)
    label_cyto = cyto_objects(cleared_cells, cleared_nuc, object_boarders,n,masks_save_path)
    cyto_props, nuc_props, cell_props = get_measurements(label_cyto,cleared_cells,cleared_nuc,center_proj,n)
    cyto_dfs.append(cyto_props)
    nuc_dfs.append(nuc_props)
    cell_dfs.append(cell_props)

df_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/"
merged_cyto = concat_measurements(cyto_dfs)
merged_cyto.to_csv(os.path.join(df_save_path,"Cytoplasm_Measurements.csv"))
merged_nuc = concat_measurements(nuc_dfs)
merged_nuc.to_csv(os.path.join(df_save_path,"Nuclei_Measurements.csv"))
merged_cell = concat_measurements(cell_dfs)
merged_cell.to_csv(os.path.join(df_save_path,"Cell_Measurements.csv"))


In [ ]:
i = randint(0,len(files))
img = sk.io.imread(files[i])
img.shape

In [ ]:
avg_img = np.mean(img,axis=1)
avg_img.shape

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(avg_img,channel_axis=0)

In [ ]:
model = models.CellposeModel(gpu=True)

In [ ]:
cell_masks, _, _ = model.eval(avg_img, normalize=True)
nuc_masks, _, _ = model.eval(avg_img[0,...], normalize=True)

In [ ]:
cell_masks, _, _ = model.eval(avg_img, normalize=True, rescale=0.5, resample=True, min_size=550)


In [ ]:
nuc_masks, _, _ = model.eval(avg_img[0,...], normalize=True, rescale=0.5, resample=True, min_size=550, cellprob_threshold=-1.0)

In [ ]:
v.add_labels(cell_masks)
v.add_labels(nuc_masks)

In [ ]:
object_boarders = sk.segmentation.find_boundaries(cell_masks,mode="inner")

In [ ]:
def clear_object_boarders(cell_masks,nuc_masks):
    cleared_cells, _, _ = sk.segmentation.relabel_sequential(sk.segmentation.clear_border(cell_masks))
    binary_cells = np.array(cleared_cells > 0,dtype="uint8")
    cleared_nuc, _, _ = sk.segmentation.relabel_sequential((binary_cells*nuc_masks))
    return cleared_cells, cleared_nuc

In [ ]:
def cyto_objects(cells, nuc, boundaries):
    binary_cells = np.array(cells>0,dtype='int8')
    binary_nuc = np.array(nuc>0,dtype='int8')
    erode_nuc = sk.morphology.erosion(binary_nuc,sk.morphology.footprint_rectangle((3,3)))
    #inter = (binary_cells - erode_nuc)
    binary_cyto = sk.exposure.rescale_intensity(((binary_cells - erode_nuc) - boundaries),in_range='image',out_range=(0,1)).astype(np.uint8)
    label_cyto = binary_cyto*cells
    return label_cyto

In [ ]:
def get_measurements(label_cyto,label_cells,label_nuc,img,name):
    props = ["label","intensity_median"]
    reshape_img = np.moveaxis(img,0,-1)
    cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_img,properties=props))
    nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_nuc,reshape_img,properties=props))
    cell_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cells,reshape_img,properties=props))
    cyto_props = cyto_props.add_prefix("Cyto_")
    nuc_props = nuc_props.add_prefix("Nuc_")
    cell_props = cell_props.add_prefix("Cell_")
    cyto_props["Nur77_Nuc_Cyto_Ratio"] = nuc_props["Nuc_intensity_median-1"]/cyto_props["Cyto_intensity_median-1"]
    cyto_props["Image"] = name
    nuc_props["Image"] = name
    cell_props["Image"] = name
    return cyto_props, nuc_props, cell_props

In [ ]:
def concat_measurements(dfs):
    return pd.concat(dfs,ignore_index=True)

In [ ]:
cleared_cells, cleared_nuc = clear_object_boarders(cell_masks,nuc_masks)

In [ ]:
label_cyto = cyto_objects(cleared_cells, cleared_nuc, object_boarders)

In [ ]:
v.add_labels(cleared_cells)
v.add_labels(cleared_nuc)
v.add_labels(label_cyto)

In [ ]:
df = get_measurements(label_cyto,cleared_cells,cleared_nuc,avg_img,names[i])

In [ ]:
df

In [ ]:
#measurements
props = ["label","intensity_median"]
reshape_avg = np.moveaxis(avg_img,0,-1)
cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_avg,properties=["label","intensity_median"]))
nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(nuc_masks,reshape_avg,properties=["label","intensity_median"]))


In [ ]:
cyto_props = cyto_props.add_prefix("Cyto_")
nuc_props = nuc_props.add_prefix("Nuc_")

In [ ]:
nuc_props.columns

In [ ]:
import seaborn as sns

In [ ]:
sns.swarmplot(nuc_props,x="Nuc_intensity_median-1",hue="Nuc_label")

In [ ]:
nuc_ints_map = sk.util.map_array(nuc_masks,nuc_props["label"],nuc_props["intensity_median-1"])

In [ ]:
v.add_image(nuc_ints_map)